<a href="https://colab.research.google.com/github/alejocastroangulo/telecom-analysis/blob/main/S7_Version_Estudiante_Project_ConnectaTel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis ConnectaTel

El objetivo es evaluar el **comportamiento de los clientes** de una empresa de telecomunicaciones en Latinoamérica, ConnectaTel.

Trabajaremos con información registrada **hasta el año 2024**, lo cual permitirá analizar el comportamiento del negocio dentro de ese periodo.

Para ello usaremos tres datasets:  

- **plans.csv** → información de los planes actuales (precio, minutos incluidos, GB incluidos, costo por extra)  
- **users.csv** → información de los clientes (edad, ciudad, fecha de registro, plan, churn)  
- **usage.csv** → detalle del **uso real** de los servicios (llamadas y mensajes)  


---
## 🧩 Paso 1: Cargar y explorar

Antes de limpiar o combinar los datos, es necesario **familiarizarte con la estructura de los tres datasets**.  
En esta etapa, validarás que los archivos se carguen correctamente, conocerás sus columnas y tipos de datos, y detectarás posibles inconsistencias.

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:**  
Tener los **3 datasets listos en memoria**, entender su contenido y realizar una revisión preliminar.

**Instrucciones:**  
- Importa las librerías necesarias (por ejemplo `pandas`, `seaborn`, `matplotlib.pyplot`)
- Carga los archivos CSV usando `pd.read_csv()`:
  - **`/datasets/plans.csv`**  
  - **`/datasets/users_latam.csv`**  
  - **`/datasets/usage.csv`**  
- Guarda los DataFrames en las variables: `plans`, `users`, `usage`.  
- Muestra las primeras filas de cada DataFrame usando `.head()`.


In [1]:
# importar librerías
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
plans = pd.read_csv('/datasets/plans.csv')
users = pd.read_csv('/datasets/users_latam.csv')
usage = pd.read_csv('/datasets/usage.csv')

In [ ]:
# mostrar las primeras filas de plans
plans.head()

In [ ]:
# mostrar las primeras 5 filas de users
users.head(5)

In [ ]:
# mostrar las primeras 5 filas de usage
usage.head(5)

### 1.2 Exploración de la estructura de los datasets

**🎯 Objetivo:**  
Conocer la **estructura de cada dataset**, revisar cuántas filas y columnas tienen, identificar los **tipos de datos** de cada columna y detectar posibles **inconsistencias o valores nulos** antes de iniciar el análisis.

**Instrucciones:**  
- Revisa el **número de filas y columnas** de cada dataset usando `.shape`.  
- Usa `.info()` en cada DataFrame para obtener un **resumen completo** de columnas, tipos de datos y valores no nulos.  

In [ ]:
# revisar el número de filas y columnas de cada dataset
print("plans", plans.shape)
print("users", users.shape)
print("usage", usage.shape)

In [ ]:
# inspección de plans con .info()
plans.info()

In [ ]:
# inspección de users con .info()
users.info()

In [ ]:
# inspección de usage con .info()
usage.info()

---

## 🧩Paso 2: Identificación de problemas de calidad de datos

### 2.1 Revisión de valores nulos

**🎯 Objetivo:**  
Detectar la presencia y magnitud de valores faltantes para evaluar si afectan el análisis o requieren imputación/eliminación.

**Instrucciones:**  
- Cuenta valores nulos por columna para cada dataset.
- Calcula la proporción de nulos por columna para cada dataset.

El dataset `plans` solamente tiene 2 renglones y se puede observar que no tiene ausentes, por ello no necesita exploración adicional.

<br>
<details>
<summary>Haz clic para ver la pista</summary>
Usa `.isna().sum()` para contar valores nulos y usa `.isna().mean()` para calcular la proporción.

In [ ]:
# cantidad de nulos para users
print(users.isna().sum())# Cantidad de valores nulos
print(users.isna().mean()) # Proporción de valores nulos

In [ ]:
# cantidad de nulos para usage
print(usage.isna().sum())
print(usage.isna().mean()) # Proporción de valores nulos

✍️ **Comentario**:

Haz doble clic en este bloque y escribe tu diagnóstico al final del bloque. Incluye qué ves y que acción recomendarías para cada caso.

💡 **Nota:** Justifica tus decisiones brevemente (1 línea por caso).
* Hint:
 - Si una columna tiene **más del 80–90% de nulos**, normalmente se **ignora o elimina**.  
 - Si tiene **entre 5% y 30%**, generalmente se **investiga para imputar o dejar como nulos**.  
 - Si es **menor al 5%**, suele ser un caso simple de imputación o dejar como nulos.

 ---

**Valores nulos**  
- ¿Qué columnas tienen valores faltantes y en qué proporción?  city(11%) y churn_date(88%) de users, por otro lado en usage tenemos date(.1%), duration(55%) y length(44%)
- Indica qué harías: ¿imputar, eliminar, ignorar?
city(11%) - revisar para imputar o dejar nulos
churn_date(88%) - se ignora
date(.1%)  - se deja nulo
duration(55%) - revisar para imputar o dejar nulos
length(44%) - revisar para imputar o dejar nulos

### 2.2 Detección de valores inválidos y sentinels

🎯 **Objetivo:**  
Identificar sentinels: valores que no deberían estar en el dataset.

**Instrucciones:**
- Explora las columnas numéricas con **un resumen estadístico** y describe brevemente que encontraste.
- Explora las columnas categóricas **relevantes**, revisando sus valores únicos y describe brevemente que encontraste.


El dataset `plans` solamente tiene 2 renglones, por ello no necesita exploración adicional.

In [ ]:
# explorar columnas numéricas de users
print(users.describe())

- La columna `user_id` La media y la mediana son iguales lo que indica consistencia, inicia en 10 mil el folio y termina en 13999, no tiene mucho valor analítico
- La columna `age`  La edad promedio es de 33.7 años mientras la mediana está en 47 años lo que indica un posible sesgo y la edad mínima está mal -999

In [ ]:
# explorar columnas numéricas de usage
print(usage.describe())

- Las columnas `id` y `user_id` Id no tiene valor analítico mientras que user ID nos puede indicar que en promedio un usuario tiene 10 registros ya que son 4000 registros únicos
- Las columnas `duration` y `length` duration no conincide en su media con la mediana hay posible sesgo mientras que lenght presetna un outlier extremo ya que el 75% de los datos se encuentra debajo de 64.

In [ ]:
# explorar columnas categóricas de users
columnas_user = ['city', 'plan']
for col in columnas_user:
    print(f"\nColumna: {col}")
    print(users[col].value_counts(dropna=False))

- La columna `city` presenta 96 registros sin identificar
- La columna `plan` cuante solo con dos tipos de paquetes, siendo el básico el de mayor volumen

In [ ]:


# explorar columna categórica de usage
usage['type'].value_counts(dropna=False)



- La columna `type`  presenta textos y llamadas indicando que hay mayor número de textos


---
✍️ **Comentario**: Haz doble clic en este bloque y escribe tu diagnóstico. Incluye qué ves y que acción recomendarías para cada caso.

**Valores inválidos o sentinels**  
- ¿En qué columnas encontraste valores inválidos o sentinels? en city encontramos valores inválidos
- ¿Qué acción tomarías?  iterar sobre los inválidos para poder proceder al análisis

### 2.3 Revisión y estandarización de fechas

**🎯 Objetivo:**  
Asegurar que las columnas de fecha estén correctamente formateadas y detectar años fuera de rango que indiquen errores de captura.

**Instrucciones:**  
- Convierte las columnas de fecha a tipo fecha y asegurate de que el código sea a prueba de errores.  
- Revisa cuántas veces aparece cada año.
- Identifica fechas imposibles (ej. años futuros o negativos).

Toma en cuenta que tenemos datos registrados hasta el año 2024.

In [ ]:


# Convertir a fecha la columna `reg_date` de users
users['reg_date'] = pd.to_datetime(users['reg_date'])
users['reg_date'].dtype
users['reg_date'].isna().sum()
users['reg_date'].describe()



In [ ]:
# Convertir a fecha la columna `date` de usage
usage['date'] = pd.to_datetime(usage['date'])
usage['date'].isna().sum()

In [ ]:
# Revisar los años presentes en `reg_date` de users
users['reg_date'].dt.year.unique()

En `reg_date`, existen 4 años diferentes y se salta del 2024 al 2026

In [ ]:
# Revisar los años presentes en `date` de usage
usage['date'].dt.year.unique()

En `date`, solo existe el 2024 y valores inválidos

✍️ **Comentario**: escribe tu diagnóstico, e incluye **qué acción recomendarías** para cada caso:

**Fechas fuera de rango**  
- ¿Aparecen años imposibles? aparece el 2026 en users pero el año no ha finalizado y en usage aparecen valores no válidos0 (años muy viejos o sin transcurrir al momento de guardar los datos)
- ¿Qué harías con ellas? iterar para analizar

---

## 🧩Paso 3: Limpieza básica de datos

### 3.1 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Aplicar reglas de limpieza para reemplazar valores sentinels y corregir fechas imposibles.

**Instrucciones:**  
- En `age`, reemplaza el sentinel **-999** con la mediana.
- En `city`, reemplaza el sentinel `"?"` por valores nulos (`pd.NA`).  
- Marca como nulas (`pd.NA`) las fechas fuera de rango.

In [ ]:
# Reemplazar -999 por la mediana de age

age_mediana = users.loc[users['age'] != -999, 'age'].median()
users['age'] = users['age'].replace(-999, age_mediana)


# Verificar cambios
users['age'].describe()

In [ ]:

# Reemplazar ? por NA en city
users['city'] = users['city'].replace('?', np.nan)

# Verificar cambios
users['city'].isna().sum()

In [ ]:

# Marcar fechas futuras como NA para reg_date
users.loc[users['reg_date'] > pd.Timestamp.today(), 'reg_date'] = pd.NaT

# Verificar cambios
users['reg_date'].describe()
users['reg_date'].isna().sum()



### 3.2 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Decidir qué hacer con los valores nulos según su proporción y relevancia.

**Instrucciones:**
- Verifica si los nulos en `duration` y `length` son **MAR**(Missing At Random) revisando si dependen de la columna `type`.  
  Si confirmas que son MAR, **déjalos como nulos** y justifica la decisión.

In [ ]:

# Verificación MAR en usage (Missing At Random) para duration


usage['duration_missing'] = usage['duration'].isna()
usage.groupby('type')['duration_missing'].mean()




In [ ]:

# Verificación MAR en usage (Missing At Random) para length
usage['length_missing'] = usage['length'].isna()
usage.groupby('type')['length_missing'].mean()




Haz doble clic aquíy escribe que tu diagnostico de nulos en `duration` y `length`  duration no se aplica para textos ya que esto sno tienen una duración mientras para las llamadas si hay duración por lo que se indica un patrón MAR y no un error en los datos

---

## 🧩Paso 4: Summary statistics de uso por usuario


### 4.1 Agrupación por comportamiento de uso

🎯**Objetivo**: Resumir las variables clave de la tabla `usage` **por usuario**, creando métricas que representen su comportamiento real de uso histórico.

**Instrucciones:**:
1. Construye una tabla agregada de `usage` por `user_id` que incluya:
- número total de mensajes  
- número total de llamadas  
- total de minutos de llamadas

2. Renombra las columnas para que tengan nombres claros:  
- `cant_mensajes`  
- `cant_llamadas`  
- `cant_minutos_llamada`
3. Combina esta tabla con `users`.

In [ ]:
# Columnas auxiliares
usage["is_text"] = (usage["type"] == "text").astype(int)  # conocer el total de mensajes
usage["is_call"] = (usage["type"] == "call").astype(int)  # conocer el total de llamadas

# Agrupar información por usuario
usage_agg = usage.groupby('user_id')[['is_text', 'is_call']].sum().reset_index()

# observar resultado
usage_agg.head(3)

In [ ]:
# Renombrar columnas
usage_agg = usage_agg.rename(columns={
    'is_text': 'total_texts',
    'is_call': 'total_calls'
})

# observar resultado
usage_agg.head(3)

In [ ]:

# Combinar la tabla agregada con el dataset de usuarios
user_profile = users.merge(usage_agg, on='user_id', how='left')

# observar resultado
user_profile.head(5)

### 4.2 4.2 Resumen estadístico por usuario durante el 2024

🎯 **Objetivo:** Analizar las columnas numéricas y categóricas de los usuarios, para identificar rangos, valores extremos y distribución de los datos antes de continuar con el análisis.

**Instrucciones:**  
1. Para las columnas **numéricas** relevantes, obtén un resumen estadístico (media, mediana, mínimo, máximo, etc.).  
2. Para la columna **categórica** `plan`, revisa la distribución en **porcentajes** de cada categoría.

In [ ]:

# Resumen estadístico de las columnas numéricas
user_profile.describe()


In [ ]:

# Distribución porcentual del tipo de plan
user_profile['plan'].value_counts(normalize=True) * 100


---

## 🧩Paso 5: Visualización de distribuciones (uso y clientes) y outliers


### 5.1 Visualización de Distribuciones

🎯 **Objetivo:**  
Entender visualmente cómo se comportan las variables clave tanto de **uso** como de **clientes**, observar si existen diferencias según el tipo de plan, y analizar la **forma de la distribución**.

**Instrucciones:**  
Graficar **histogramas** para las siguientes columnas:  
- `age` (edad de los usuarios)
- `cant_mensajes`
- `cant_llamadas`
- `total_minutos_llamada`

Después de cada gráfico, escribe un **insight** respecto al plan y la variable, por ejemplo:  
- "Dentro del plan Premium, hay mayor proporción de..."  
- "Los usuarios Básico tienden a hacer ... llamadas y enviar ... mensajes."  o "No existe algún patrón."
- ¿Qué tipo de distribución tiene ? (simétrica, sesgada a la derecha o a la izquierda)

**Hint**  
Para cada histograma,
- Usa `hue='plan'` para ver cómo varían las distribuciones según el plan (Básico o Premium).
- Usa `palette=['skyblue','green']`
- Agrega título y etiquetas

In [ ]:
# Histograma para visualizar la edad (age)

sns.histplot(user_profile['age'], bins=20)
plt.title("Distribución de edad de los usuarios")
plt.xlabel("Edad")
plt.ylabel("Frecuencia")
plt.show()

💡Insights:
- Distribución relativamente uniforme
La frecuencia de usuarios se mantiene relativamente estable a lo largo de los distintos rangos de edad, sin concentraciones extremadamente marcadas en un grupo específico.

In [ ]:

# Histograma para visualizar la cant_mensajes

sns.histplot(user_profile['total_texts'], bins=30)


💡Insights:
1 La mayoría de los usuarios envía pocos mensajes
La distribución muestra que la mayor concentración de usuarios se encuentra aproximadamente entre 4 y 7 mensajes, lo que indica que el uso de SMS es relativamente bajo para la mayoría de los clientes.

2️ Distribución sesgada a la derecha
Se observa una cola hacia valores más altos, lo que significa que existe un pequeño grupo de usuarios que envía significativamente más mensajes que el promedio.

3️ Los SMS no parecen ser el servicio principal
Dado que el número de mensajes es relativamente bajo para la mayoría de los usuarios, esto sugiere que los SMS no son el principal canal de comunicación, posiblemente desplazados por aplicaciones de mensajería basadas en internet.

In [ ]:
# Histograma para visualizar la cant_llamadas
sns.histplot(user_profile['total_calls'], bins=30)
plt.title("Distribución de cantidad de llamadas por usuario")
plt.xlabel("Cantidad de llamadas")
plt.ylabel("Frecuencia")
plt.show()


💡Insights:
- Age:                 NO presenta outliers(presenta o no outliers)
- cant_mensajes:       SI presena outliers
- cant_llamadas:       SI presena outliers
- cant_minutos_llamada:NO presenta outliers

In [ ]:

# Calcular límites con el método IQR
columnas_limites = ['age', 'total_texts', 'total_calls']
for col in columnas_limites:
    Q1 = user_profile[col].quantile(0.25)
    Q3 = user_profile[col].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    print(f"{col}")
    print(f"Límite inferior: {limite_inferior}")
    print(f"Límite superior: {limite_superior}")
    print()


In [ ]:
# Revisa los limites superiores y el max, para tomar la decisión de mantener los outliers o no
user_profile[columnas_limites].describe()

💡Insights:
- cant_mensajes: mantener o no outliers, porqué? Mantener, porque los outliers no son valores imposibles ni errores de datos
- cant_llamadas: mantener o no outliers, porqué?  Mantener, porque algunos usuarios realizan muchas más llamadas que otros, no hay error
- cant_minutos_llamada: mantener o no outliers, porqué? Mantener ya que son valores posibles reales de clientes quehacen llamadas de mayor duración

---

## 🧩Paso 6: Segmentación de Clientes

### 6.1 Segmentación de Clientes Por Uso

🎯 **Objetivo:** Clasificar a cada usuario en un grupo de uso (Bajo uso, Uso medio, Alto uso) basándose en la cantidad de llamadas y mensajes registrados.

**Instrucciones:**  
- Crea una nueva columna llamada `grupo_uso` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones de llamadas y mensajes y asigna:
  - `'Bajo uso'` cuando llamadas < 5 y mensajes < 5
  - `'Uso medio'` cuando llamadas < 10 y mensajes < 10
  - `'Alto uso'` para el resto de casos

In [ ]:

# Crear columna grupo_uso
user_profile['grupo_uso'] = 'Alto uso'

user_profile.loc[
    (user_profile['total_calls'] < 5) & (user_profile['total_texts'] < 5),
    'grupo_uso'
] = 'Bajo uso'

user_profile.loc[
    (user_profile['total_calls'] < 10) & (user_profile['total_texts'] < 10) &
    (user_profile['grupo_uso'] != 'Bajo uso'),
    'grupo_uso'
] = 'Uso medio'




In [ ]:

# Verificar resultado
user_profile.head()


### 6.2 Segmentación de Clientes Por Edad

🎯 **Objetivo:**: Clasificar a cada usuario en un grupo por **edad**.

**Instrucciones:**  
- Crea una nueva columna llamada `grupo_edad` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones y asigna:
  - `'Joven'` cuando age < 30
  - `'Adulto'` cuando age < 60
  - `'Adulto Mayor'` para el resto de casos

In [ ]:

# Crear columna grupo_edad

user_profile['grupo_edad'] = 'Adulto Mayor'

user_profile.loc[user_profile['age'] < 30, 'grupo_edad'] = 'Joven'

user_profile.loc[
    (user_profile['age'] >= 30) & (user_profile['age'] < 60),
    'grupo_edad'
] = 'Adulto'


In [ ]:
# verificar cambios
user_profile.head()

### 6.3 Visualización de la Segmentación de Clientes

🎯 **Objetivo:** Visualizar la distribución de los usuarios según los grupos creados: **grupo_uso** y **grupo_edad**.

**Instrucciones:**  
- Crea dos gráficos para las variables categóricas `grupo_uso` y `grupo_edad`.
- Agrega título y etiquetas a los ejes en cada gráfico.

In [ ]:
# Visualización de los segmentos por uso
sns.countplot(data=user_profile, x='grupo_uso')
plt.title("Distribución de usuarios por nivel de uso")
plt.xlabel("Grupo de uso")
plt.ylabel("Número de usuarios")
plt.show()

In [ ]:
# Visualización de los segmentos por edad
sns.countplot(data=user_profile, x='grupo_edad')
plt.title("Distribución de usuarios por grupo de edad")
plt.xlabel("Grupo de edad")
plt.ylabel("Número de usuarios")
plt.show()


---
## 🧩Paso 7: Insight Ejecutivo para Stakeholders

🎯 **Objetivo:** Traducir los hallazgos del análisis en conclusiones accionables para el negocio, enfocadas en segmentación, patrones de uso y oportunidades comerciales.

**Preguntas a responder:**
- ¿Qué problemas tenían originalmemte los datos?¿Qué porcentaje, o cantidad de filas, de esa columna representaban?


- ¿Qué segmentos de clientes identificaste y cómo se comportan según su edad y nivel de uso?  
- ¿Qué segmentos parecen más valiosos para ConnectaTel y por qué?  
- ¿Qué patrones de uso extremo (outliers) encontraste y qué implican para el negocio?


- ¿Qué recomendaciones harías para mejorar la oferta actual de planes o crear nuevos planes basados en los segmentos y patrones detectados?

✍️ **Escribe aquí tu análisis ejecutivo:**

### Análisis ejecutivo

⚠️ **Problemas detectados en los datos**

Durante la revisión de los datos se identificaron algunos problemas de calidad. En la columna duration aparecieron muchos valores faltantes en los registros de tipo text. Esto no representa un error, ya que los mensajes de texto no tienen duración.

También se encontraron valores incorrectos en la columna age, como -999, que se utilizaban para representar datos faltantes. Estos valores fueron reemplazados por la mediana de edad para evitar afectar el análisis.

Además, en la columna city se detectaron valores como "?", los cuales fueron reemplazados por valores nulos. Finalmente, se encontraron algunas fechas futuras en reg_date, las cuales se corrigieron marcándolas como valores faltantes.


🔍 **Segmentos por Edad**



Se identificaron tres grupos principales de clientes:

Jóvenes (menores de 30 años)

Adultos (entre 30 y 59 años)

Adultos mayores (60 años o más)

El grupo más grande corresponde a los adultos, lo que indica que la mayor parte de los clientes pertenece a este segmento. Los jóvenes muestran mayor variabilidad en su uso del servicio, mientras que los adultos mayores tienden a tener un uso más moderado.


📊 **Segmentos por Nivel de Uso**
Los usuarios se clasificaron según su nivel de uso en tres grupos:

Bajo uso

Uso medio

Alto uso

La mayoría de los clientes se encuentra en los grupos de bajo y uso medio, lo que indica que utilizan el servicio de forma moderada. Sin embargo, también existe un grupo más pequeño de usuarios con alto uso, que realizan más llamadas y envían más mensajes que el promedio.

➡️ Esto sugiere que ...


💡 **Recomendaciones**

Con base en los resultados del análisis, se recomienda crear planes diferenciados según el nivel de uso de los clientes. Por ejemplo, planes básicos para usuarios de bajo consumo y planes más completos para usuarios intensivos.

También sería útil identificar a los clientes que frecuentemente superan los límites de su plan actual y ofrecerles planes más adecuados a su nivel de uso, lo que podría mejorar la satisfacción del cliente y aumentar los ingresos de la empresa.

---

## 🧩Paso 8 Cargar tu notebook y README a GitHub

🎯 **Objetivo:**  
Entregar tu análisis de forma **profesional**, **documentada** y **versionada**, asegurando que cualquier persona pueda revisar, ejecutar y entender tu trabajo.



### Opción A : Subir archivos desde la interfaz de GitHub (UI)

1. Descarga este notebook (`File → Download .ipynb`).  
2. Entra a tu repositorio en GitHub (por ejemplo `telecom-analysis` o `sprint7-final-project`).  
3. Sube tu notebook **Add file → Upload files**.  

---

### Opción B : Guardar directo desde Google Colab

1. Abre tu notebook en Colab.  
2. Ve a **File → Save a copy in GitHub**.  
3. Selecciona el repositorio y la carpeta correcta (ej: `notebooks/`).  
4. Escribe un mensaje de commit claro, por ejemplo:  
    - `feat: add final ConnectaTel analysis`
    - `agregar version final: Análisis ConnectaTel`
5. Verifica en GitHub que el archivo quedó en el lugar correcto y que el historial de commits se mantenga limpio.

---

Agrega un archivo `README.md` que describa de forma clara:
- el objetivo del proyecto,  
- los datasets utilizados,  
- las etapas del análisis realizadas,  
- cómo ejecutar el notebook (por ejemplo, abrirlo en Google Colab),  
- una breve guía de reproducción.
---

Link a repositorio público del proyecto: `LINK a tu repo aquí`